# Data Extraction and Storage

## 1. Steps to collect airport data using API and convert into SQL

In [ ]:
import time
import pandas as pd

In [ ]:
#Connection to the url with key
import requests
API_HOST = "aerodatabox.p.rapidapi.com"
API_KEY = "ae6852a1abmshff7e7a387a1f740p1da32ejsn8af8b9de9fbd"

headers = {
	"x-rapidapi-key": API_KEY,
	"x-rapidapi-host": API_HOST
} 

In [ ]:
#Defining a function with icao code
def fetch_airport_by_icao(icao_code):
    url=f"https://aerodatabox.p.rapidapi.com/airports/icao/{icao_code}"
    airport_response=requests.get(url,headers=headers)
    time.sleep(1)
    return airport_response.json()
    

In [ ]:
my_airports=["EGLL", "KJFK", "OMDB", "VIDP", "VOBL", "EGCC", "EDDM", "CYVR", "RJOO", "WMKP", "VTCC", "EHAM", "LFPG", "WSSS", "YSSY"]

airport_all={}

for icao in my_airports:
    try:
        airport_data=fetch_airport_by_icao(icao)
        airport_all[icao]=airport_data
        print(airport_data)
    except Exception as e:
        print(f"Error loading airport {icao}: {e}")

In [ ]:


def extract_airport_info(airport_data):
    extracted_info = {
        'icao_code': airport_data.get('icao'),
        'iata_code': airport_data.get('iata'),
        'name': airport_data.get('fullName'),
        'city': airport_data.get('municipalityName'),
        'country': airport_data.get('country', {}).get('name'),
        'continent': airport_data.get('continent', {}).get('name'),
        'latitude': airport_data.get('location', {}).get('lat'),
        'longitude': airport_data.get('location', {}).get('lon'),
        'timezone': airport_data.get('timeZone')
    }
    return extracted_info

In [ ]:
extracted_airports_data = []
for icao, data in airport_all.items():
    extracted_airports_data.append(extract_airport_info(data))

# Create a pandas DataFrame from the extracted data
airport_df = pd.DataFrame(extracted_airports_data)
display(airport_df)

In [2]:
#To make a connection to mysql
import mysql.connector 

connection = mysql.connector.connect(
    host = "Localhost",
    user = "root",
    password = "12345",
    )
cursor =  connection.cursor()
cursor

In [ ]:
query="create database if not exists AeroDataBox"
cursor.execute(query)
connection.commit()

In [3]:
query = "use AeroDataBox"
cursor.execute(query)

In [ ]:
query = """create table if not exists Airport(airport_id int primary key auto_increment, icao_code varchar(60) UNIQUE, iata_code varchar(60) UNIQUE, name varchar(60), city varchar(60), country varchar(60), continent varchar(60), latitude float, longitude float, timezone varchar(60))"""
cursor.execute(query)
connection.commit()

In [ ]:
columns = ",".join(airport_df.columns)
placeholders = ",".join(["%s"]*len(airport_df.columns))
query = f"insert ignore into Airport({columns}) values({placeholders})"
tuple_data = []
for i in airport_df.index:
    row = list(airport_df.loc[i].values)
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

## 2. Steps to collect flight data using API and convert into SQL

In [ ]:
#Defining a function with icao code
def fetch_flight_by_icao(icao_code):
    url=f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao_code}/2026-05-17T20:00/2026-05-18T08:00"
    querystring = {"withLeg":"true","direction":"Both","withCancelled":"true","withCodeshared":"true","withCargo":"true","withPrivate":"true","withLocation":"false"}

    flight_response=requests.get(url,headers=headers, params=querystring)
    time.sleep(1)
    return flight_response.json()

In [ ]:
def extract_flight_info(flight_data, flight_type, airport_icao):
    flight_id = flight_data.get('number', '') + '_' + flight_data.get(flight_type, {}).get('scheduledTime', {}).get('utc', '')

    # Determine origin and destination based on flight type, ensuring ICAO codes
    origin_code = ''
    destination_code = ''
    if flight_type == 'departure':
        origin_code = airport_icao # The airport from which the flights were fetched
        destination_code = flight_data.get('arrival', {}).get('airport', {}).get('icao')
    elif flight_type == 'arrival':
        origin_code = flight_data.get('departure', {}).get('airport', {}).get('icao')
        destination_code = airport_icao # The airport to which the flights were fetched

    return {
        'flight_id': flight_id,
        'flight_number': flight_data.get('number'),
        'aircraft_registration': flight_data.get('aircraft', {}).get('reg'),
        'origin_icao': origin_code,
        'destination_icao': destination_code,
        'scheduled_departure': flight_data.get('departure', {}).get('scheduledTime', {}).get('utc', ''),
        'actual_departure': flight_data.get('departure', {}).get('revisedTime', {}).get('utc', flight_data.get('departure', {}).get('scheduledTime', {}).get('utc')),#fallback extraction
        'scheduled_arrival': flight_data.get('arrival', {}).get('scheduledTime', {}).get('utc'),
        'actual_arrival': flight_data.get('arrival', {}).get('revisedTime', {}).get('utc', flight_data.get('arrival', {}).get('scheduledTime', {}).get('utc')),
        'status': flight_data.get('status'),
        'airline_code': flight_data.get('airline', {}).get('iata')
    }



In [ ]:
all_flights_data = []
list_icao = airport_df['icao_code']
flights_all={}
for icao in list_icao:
    try:
        airflight_data=fetch_flight_by_icao(icao)
        flight_all[icao] = airflight_data
        print(flight_all[icao])
    except Exception as e:
        print(f"Error loading flight {icao}: {e}")        
    

In [ ]:
for icao in list_icao:
    # Get current airport data
        curr_airport_data = flight_all[icao]
    # Extract departures
        if 'departures' in curr_airport_data:
            for flight in curr_airport_data.get('departures',[])[:20]:
                all_flights_data.append(extract_flight_info(flight, 'departure', icao)) 

# Extract arrivals
        if 'arrivals' in curr_airport_data:
            for flight in curr_airport_data.get('arrivals',[])[:20]:
                all_flights_data.append(extract_flight_info(flight, 'arrival', icao))
    
# Create DataFrame
flights_df = pd.DataFrame(all_flights_data)
display(flights_df)


In [ ]:
flights_df=flights_df.fillna("None")
flights_df.info()

In [ ]:
query="""create table if not exists flight(flight_id varchar(60) PRIMARY KEY, flight_number varchar(60), aircraft_registration varchar(60), origin_icao varchar(60), destination_icao varchar(60), scheduled_departure varchar(60), actual_departure varchar(60), scheduled_arrival varchar(60), actual_arrival varchar(60), status varchar(60), airline_code varchar(60))"""
cursor.execute(query)

In [ ]:
columns = ",".join(flights_df.columns)
placeholders = ",".join(["%s"]*len(flights_df.columns))
query = f"insert into flight({columns}) values({placeholders})"
tuple_data = []
for i in flights_df.index:
    row = list(flights_df.loc[i].values)
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

In [14]:
query = """DELETE FROM flight WHERE scheduled_departure IS NULL OR scheduled_departure = '' OR scheduled_departure = 'None'
            OR actual_departure IS NULL OR actual_departure = '' OR actual_departure = 'None'
            OR scheduled_arrival IS NULL OR scheduled_arrival = '' OR scheduled_arrival = 'None'
            OR actual_arrival IS NULL OR actual_arrival = '' OR actual_arrival = 'None'"""
cursor.execute(query)
connection.commit()

In [ ]:
list_aircraft=flights_df['aircraft_registration'].unique().tolist()#294 entries
cleanlist_aircraft=[x for x in list_aircraft if x != "None"]
cleanlist_aircraft

## 3. Steps to collect aircraft data using API and convert into SQL

In [ ]:
#Defining a function with icao code
def fetch_aircraft_by_reg(reg_code):
    url=f"https://aerodatabox.p.rapidapi.com/aircrafts/reg/{reg_code}"
    aircraft_response=requests.get(url,headers=headers)
    time.sleep(1)
    return aircraft_response.json()
    

In [ ]:
aircraft_all={}

for REG_code in cleanlist_aircraft[:100]:
    try:
        aircraft_data=fetch_aircraft_by_reg(REG_code)
        aircraft_all[REG_code]=aircraft_data
        print(REG_code, aircraft_all[REG_code])
    except Exception as e:
        print(f"Error loading airport {REG_code}: {e}")

In [ ]:
import pandas as pd

def extract_aircraft_info(aircraft_data):
    extracted_aircraft_info = {
        'aircraft_code': aircraft_data.get('reg'),
        'aircraft_type': aircraft_data.get('typeName'),
        'serial': aircraft_data.get('serial'),
        'model': aircraft_data.get('model'),
        'airline': aircraft_data.get('airlineName'),
        'icao_code': aircraft_data.get('icaoCode'),
        'engine': aircraft_data.get('engineType'),
        'registration': aircraft_data.get('id')         
        }
    return extracted_aircraft_info

In [ ]:
extracted_aircrafts_data = []
for aircraft_code, data in aircraft_all.items():
        extracted_aircrafts_data.append(extract_aircraft_info(data))

# Create a pandas DataFrame from the extracted data
aircraft_df = pd.DataFrame(extracted_aircrafts_data)

aircraft_df_dropped = aircraft_df.dropna() #drops rows with null values since NaN and NaT are not accepteble in sql
display(aircraft_df_dropped)#dropped 2 rows

In [ ]:
query = """create table if not exists Aircraft(aircraft_id int primary key AUTO_INCREMENT, aircraft_code varchar(60) UNIQUE, aircraft_type varchar(60), serial varchar(60), model varchar(60), airline varchar(60), icao_code varchar(60), engine varchar(60), registration int unique)"""
cursor.execute(query)
connection.commit()

In [ ]:
columns = ",".join(aircraft_df_dropped.columns)
placeholders = ",".join(["%s"]*len(aircraft_df_dropped.columns))
query = f"insert into Aircraft({columns}) values({placeholders})"
tuple_data = []
for i in aircraft_df_dropped.index:
    row = list(aircraft_df_dropped.loc[i].values)
    row[7]=int(row[7])
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

## 4. Steps to collect airport_delays data using API and convert into SQL

In [7]:
query = """create table if not exists airport_delays(delay_id int primary key auto_increment, flight_number varchar(60), dep_date varchar(60), sch_dep_time varchar(60), act_dep_time varchar(60), sch_arr_time varchar(60), act_arr_time varchar(60), diff_sch_act_dep varchar(60), diff_sch_act_act varchar(60), dep_status varchar(60), arr_status varchar(60))"""
cursor.execute(query)
cursor

In [17]:
query = """INSERT INTO airport_delays (flight_number, dep_date, sch_dep_time, act_dep_time, sch_arr_time, act_arr_time,
                                       diff_sch_act_dep,  diff_sch_act_act, dep_status, arr_status)
            SELECT temp.flight_number, temp.dep_date, temp.sch_dep_time, temp.act_dep_time, temp.sch_arr_time, temp.act_arr_time, 
                   temp.diff_sch_act_dep, temp.diff_sch_act_act, 
            CASE
               WHEN temp.diff_sch_act_dep > 0 THEN 'delayed'
               ELSE 'on_time'
            END AS dep_status,
            CASE
                WHEN temp.diff_sch_act_act > 0 THEN 'delayed'
                ELSE 'on_time'
            END AS arr_status
            FROM (SELECT flight.flight_number, flight.actual_departure, flight.actual_arrival,
            DATE(STR_TO_DATE(flight.scheduled_departure, '%Y-%m-%d %H:%iZ')) AS dep_date,
            DATE_FORMAT(STR_TO_DATE(flight.scheduled_departure, '%Y-%m-%d %H:%iZ'), '%H:%i') AS sch_dep_time,
            DATE_FORMAT(STR_TO_DATE(flight.actual_departure, '%Y-%m-%d %H:%iZ'), '%H:%i') AS act_dep_time,
            DATE_FORMAT(STR_TO_DATE(flight.scheduled_arrival, '%Y-%m-%d %H:%iZ'), '%H:%i') AS sch_arr_time,    
            DATE_FORMAT(STR_TO_DATE(flight.actual_arrival, '%Y-%m-%d %H:%iZ'), '%H:%i') AS act_arr_time,
            TIMESTAMPDIFF(MINUTE,
                STR_TO_DATE(flight.scheduled_departure, '%Y-%m-%d %H:%iZ'),
                STR_TO_DATE(flight.actual_departure, '%Y-%m-%d %H:%iZ')) AS diff_sch_act_dep,
            TIMESTAMPDIFF(MINUTE, STR_TO_DATE(flight.scheduled_arrival, '%Y-%m-%d %H:%iZ'),
                STR_TO_DATE(flight.actual_arrival, '%Y-%m-%d %H:%iZ')) AS diff_sch_act_act
            FROM flight) temp"""
cursor.execute(query)
connection.commit()